In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# --- Configuration ---
# Make sure these match your previous script exactly
model_types = ["plsr", "xgb", "lgbm", "rf"]
# base_dir = "models" 
eval_dir = "evaluations"
os.makedirs(eval_dir, exist_ok=True)

# Helper function for Mean Percentage Error (MPE)
def mean_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Prevent division by zero
    non_zero_mask = y_true != 0
    if not np.any(non_zero_mask):
        return 0.0 # Fallback if all true values are exactly 0
    return np.mean((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask]) * 100

# Get all unique targets across your blueprints
all_targets = set()
for m_type in model_types:
    all_targets.update(blueprints[m_type]['Feature'].unique())

print("Starting Evaluation Pipeline...")

for target in all_targets:
    print(f"\n========== EVALUATING {target} ==========")
    
    # ---------------------------------------------------------
    # 1. RECONSTRUCT THE DATASET
    # ---------------------------------------------------------
    # We need to apply the same masking to get the exact same split.
    # We will assume the most complex config (Spectral + pH + EC) 
    # was used to ensure all models have the necessary columns available.
    
    prediction_columns = base_spectral_columns.copy() + ["p1.pH.index", "p1.EC.ds_m"]
    maskpH = df["p1.pH.index"].notna()
    maskEC = df["p1.EC.ds_m"].notna()
    target_mask = df[target].notna()
    
    final_mask = target_mask & maskpH & maskEC
    
    if final_mask.sum() == 0:
        print(f"⚠️ Skipping {target} - Not enough overlapping data.")
        continue
        
    y_clean = df.loc[final_mask, target]
    X_clean = df.loc[final_mask, prediction_columns]
    
    # EXACT same split
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_clean, test_size=0.2, random_state=42
    )
    
    # ---------------------------------------------------------
    # 2. LOAD MODELS & GENERATE PREDICTIONS
    # ---------------------------------------------------------
    test_predictions = {}
    train_cv_predictions = {}
    
    for m_type in model_types:
        file_path = os.path.join(base_dir, m_type, f"{target}_model.pkl")
        
        if not os.path.exists(file_path):
            print(f"  Missing {m_type.upper()} model for {target}, skipping this model.")
            continue
            
        # Load the saved pipeline package
        save_package = joblib.dump(file_path)
        pipeline = save_package['pipeline']
        required_features = save_package['features_required']
        
        # Subset the X data to only what this specific model requires
        X_train_model = X_train[required_features]
        X_test_model = X_test[required_features]
        
        # Predict on Test Set (for final evaluation)
        test_predictions[m_type.upper()] = pipeline.predict(X_test_model)
        
        # Generate Cross-Validated Predictions on Train Set (for the Stacker)
        # cv=5 means 5-fold cross-validation
        train_cv_predictions[m_type.upper()] = cross_val_predict(
            pipeline, X_train_model, y_train, cv=5, n_jobs=-1
        )
    
    if not test_predictions:
        print(f"  No models found for {target}. Moving to next.")
        continue

    # ---------------------------------------------------------
    # 3. TRAIN THE STACKER (META-MODEL)
    # ---------------------------------------------------------
    print("  Training Linear Stacker...")
    # Convert prediction dictionaries to 2D numpy arrays (Features for the meta-model)
    X_train_meta = pd.DataFrame(train_cv_predictions)
    X_test_meta = pd.DataFrame(test_predictions)
    
    # Train Linear Regression on the unbiased CV predictions
    stacker = LinearRegression()
    stacker.fit(X_train_meta, y_train)
    
    # Get final stacked predictions for the test set
    test_predictions['STACKER'] = stacker.predict(X_test_meta)
    
    # ---------------------------------------------------------
    # 4. CALCULATE METRICS
    # ---------------------------------------------------------
    models_to_eval = list(test_predictions.keys())
    metrics_data = []
    
    for model_name in models_to_eval:
        preds = test_predictions[model_name]
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        mpe = mean_percentage_error(y_test, preds)
        
        metrics_data.append({
            'Model': model_name,
            'R2': r2,
            'MAE': mae,
            'MPE': mpe
        })
        
    metrics_df = pd.DataFrame(metrics_data)
    
    # ---------------------------------------------------------
    # 5. GENERATE DASHBOARD
    # ---------------------------------------------------------
    print("  Generating Dashboard...")
    sns.set_theme(style="whitegrid")
    # Create a 2-row layout: Top row for scatters, bottom row for metrics
    fig = plt.figure(figsize=(24, 12))
    fig.suptitle(f'Model Evaluation Dashboard: {target}', fontsize=24, fontweight='bold', y=0.98)
    
    # Subplot grid: 2 rows. Top row has 5 columns, Bottom row has 3 columns.
    gs = fig.add_gridspec(2, 15) 
    
    # --- TOP ROW: Scatter Plots ---
    for i, model_name in enumerate(models_to_eval):
        ax = fig.add_subplot(gs[0, i*3:(i+1)*3])
        preds = test_predictions[model_name]
        
        # Actual vs Predicted Scatter
        ax.scatter(y_test, preds, alpha=0.6, edgecolors='w', s=50)
        
        # 1:1 Perfect Prediction Line
        min_val = min(y_test.min(), preds.min())
        max_val = max(y_test.max(), preds.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Line')
        
        # Linear Trendline (np.polyfit)
        z = np.polyfit(y_test, preds, 1)
        p = np.poly1d(z)
        ax.plot(y_test, p(y_test), 'r-', lw=2, label='Trendline')
        
        ax.set_title(model_name, fontsize=16, fontweight='bold')
        ax.set_xlabel('Actual Values', fontsize=12)
        ax.set_ylabel('Predicted Values', fontsize=12)
        ax.legend()

    # --- BOTTOM ROW: Bar Charts ---
    # R-Squared
    ax_r2 = fig.add_subplot(gs[1, 1:5])
    sns.barplot(data=metrics_df, x='Model', y='R2', ax=ax_r2, palette='viridis')
    ax_r2.set_title('$R^2$ Score (Higher is Better)', fontsize=14)
    ax_r2.set_ylim(0, 1.0) # Standard R2 range
    
    # MAE
    ax_mae = fig.add_subplot(gs[1, 6:10])
    sns.barplot(data=metrics_df, x='Model', y='MAE', ax=ax_mae, palette='magma')
    ax_mae.set_title('Mean Absolute Error (Lower is Better)', fontsize=14)
    
    # MPE
    ax_mpe = fig.add_subplot(gs[1, 11:15])
    sns.barplot(data=metrics_df, x='Model', y='MPE', ax=ax_mpe, palette='coolwarm')
    ax_mpe.set_title('Mean Percentage Error (%)', fontsize=14)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95]) # Adjust to fit main title
    
    # Save Figure
    save_path = os.path.join(eval_dir, f"{target}_dashboard.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  -> Dashboard saved to {save_path}")

print("\nAll evaluations complete!")

NameError: name 'blueprints' is not defined